In [0]:
from pyspark.sql.functions import *

start_date = "2024-01-01"
end_date = "2025-12-01"

df = spark.sql(
    f"""
    SELECT explode(
        sequence(
            to_date('{start_date}'),
            to_date('{end_date}'),
            interval 1 month
        )
    ) AS month_start_date
    """
) 

df = (
    df
    .withColumn(
        "date_key",
        date_format(df["month_start_date"], "yyyyMMdd").cast("int")
    )
    .withColumn(
        "year",
        year("month_start_date")
    )
    .withColumn(
        "month_name",
        date_format(df["month_start_date"], "MMMM")
    )
    .withColumn(
        "month_short_name",
        date_format(df["month_start_date"], "MMM")
    )
    .withColumn(
        "quarter",
        concat(lit("Q"), quarter("month_start_date"))
    )
    .withColumn(
        "year_quarter",
        concat(
            year("month_start_date"),
            lit("-Q"),
            quarter("month_start_date")
        )
    )
) 

display(df)

month_start_date,date_key,year,month_name,month_short_name,quarter,year_quarter
2024-01-01,20240101,2024,January,Jan,Q1,2024-Q1
2024-02-01,20240201,2024,February,Feb,Q1,2024-Q1
2024-03-01,20240301,2024,March,Mar,Q1,2024-Q1
2024-04-01,20240401,2024,April,Apr,Q2,2024-Q2
2024-05-01,20240501,2024,May,May,Q2,2024-Q2
2024-06-01,20240601,2024,June,Jun,Q2,2024-Q2
2024-07-01,20240701,2024,July,Jul,Q3,2024-Q3
2024-08-01,20240801,2024,August,Aug,Q3,2024-Q3
2024-09-01,20240901,2024,September,Sep,Q3,2024-Q3
2024-10-01,20241001,2024,October,Oct,Q4,2024-Q4


In [0]:
# Save as table: 
df.write.mode("overwrite").format("delta").saveAsTable("fmcg.gold.dim_date")